In [0]:
from pyspark.sql.functions import (
    col,
    when,
    concat_ws,
    lit,
    trim
)

# Read raw Bronze table
df_raw = spark.table("final_assesment.bronze.orders_raw")

print("Raw Bronze record count:", df_raw.count())

display(df_raw)

In [0]:
df_validated = (
    df_raw
    .withColumn(
        "reject_reason",
        concat_ws(
            ", ",
            when(
                col("order_id").isNull() | (trim(col("order_id")) == ""),
                lit("Missing order_id")
            ),
            when(
                col("order_date").isNull(),
                lit("Missing order_date")
            ),
            when(
                col("customer_id").isNull() | (trim(col("customer_id")) == ""),
                lit("Missing customer_id")
            ),
            when(
                col("product_id").isNull() | (trim(col("product_id")) == ""),
                lit("Missing product_id")
            ),
            when(
                col("quantity").isNull() | (col("quantity") <= 0),
                lit("Invalid quantity")
            ),
            when(
                col("unit_price").isNull() | (col("unit_price") < 0),
                lit("Invalid unit_price")
            ),
            when(
                col("discount").isNull() | (col("discount") < 0),
                lit("Invalid discount")
            ),
            when(
                (col("quantity") * col("unit_price") < col("discount")),
                lit("Discount exceeds gross amount")
            )
        )
    )
)

display(df_validated)

In [0]:
# Separate rejected records
df_rejected = (
    df_validated
    .filter(col("reject_reason") != "")
)

# Separate valid records
df_valid = (
    df_validated
    .filter(col("reject_reason") == "")
    .drop("reject_reason")
)

print("Total raw records:", df_raw.count())
print("Rejected records:", df_rejected.count())
print("Valid records:", df_valid.count())

In [0]:
display(
    df_rejected.select(
        "order_id",
        "quantity",
        "unit_price",
        "discount",
        "reject_reason",
        "batch_id",
        "source_file"
    )
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, when

# Assign status priority
df_priority = df_valid.withColumn(
    "status_priority",
    when(col("order_status") == "Cancelled", 1)
    .when(col("order_status") == "Delivered", 2)
    .when(col("order_status") == "Shipped", 3)
    .when(col("order_status") == "Processing", 4)
    .otherwise(5)
)

# Define window for duplicate identification
window_spec = (
    Window
    .partitionBy("order_id")
    .orderBy(
        col("order_updated_at").desc(),
        col("status_priority").asc()
    )
)

# Rank records
df_ranked = df_priority.withColumn(
    "row_num",
    row_number().over(window_spec)
)

# Keep the latest/highest-priority record
df_deduplicated = (
    df_ranked
    .filter(col("row_num") == 1)
    .drop("row_num", "status_priority")
)

# Capture removed duplicate records
df_duplicates = (
    df_ranked
    .filter(col("row_num") > 1)
    .drop("row_num", "status_priority")
    .withColumn(
        "reject_reason",
        lit("REJECTED_DUE_TO_DUPLICATE")
    )
)

print("Records before deduplication:", df_valid.count())
print("Duplicate records:", df_duplicates.count())
print("Records after deduplication:", df_deduplicated.count())

In [0]:
df_final_rejected = df_rejected.unionByName(
    df_duplicates,
    allowMissingColumns=True
)

print("Total rejected records:", df_final_rejected.count())

display(df_final_rejected)

In [0]:
order_ids = [
    "ORD01009",
    "ORD01021",
    "ORD01048",
    "ORD01072",
    "ORD01103",
    "ORD01116"
]

df_selected = df_raw.filter(col("order_id").isin(order_ids)).orderBy("order_id")

print("Records found for the given order_ids:", df_selected.count())

display(df_selected)

In [0]:
(
    df_final_rejected
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("final_assesment.bronze.orders_rejected")
)

print("Rejected records saved successfully.")

In [0]:
from pyspark.sql.functions import upper, initcap

df_silver = (
    df_deduplicated

    # Trim and standardize order status
    .withColumn("order_status", upper(trim(col("order_status"))))

    # Customer standardization
    .withColumn("customer_name", initcap(trim(col("customer_name"))))
    .withColumn("customer_city", initcap(trim(col("customer_city"))))
    .withColumn("customer_state", upper(trim(col("customer_state"))))

    # Product standardization
    .withColumn("product_name", initcap(trim(col("product_name"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("brand", (trim(col("brand"))))

    # Payment standardization
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("payment_provider", (trim(col("payment_provider"))))
)

display(df_silver)

In [0]:
display(
    df_silver.select(
        "order_status",
        "customer_name",
        "customer_city",
        "customer_state",
        "product_name",
        "category",
        "brand",
        "payment_method",
        "payment_provider"
    )
)

In [0]:
silver_table = "final_assesment.silver.orders_clean"

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print("Silver table created successfully!")
print("Clean records written:", spark.table(silver_table).count())